In [1]:
import os
import pandas as pd

In [2]:
tile_size = 256
qpfolder = "/rsrch5/home/trans_mol_path/cercan/data/acformer/qpproj_dummy/"
predictions = os.path.join(qpfolder,'predictions')
parent_output = os.path.join(qpfolder,'wsi_prediction_csv')
os.makedirs(parent_output, exist_ok =True)

img_list = os.listdir(predictions)


In [5]:
def coords_from_fname (filename):
    x = int(filename.split('x=')[1].split(',')[0])
    y = int(filename.split('y=')[1].split(',')[0])

    return x, y

def removeEmptyCsv(csv_folder):
    full_csv_path_list = []
    for csv_name in os.listdir(csv_folder):
        csv_path = os.path.join(csv_folder, csv_name)
        if os.path.getsize(csv_path) > 0:
            full_csv_path_list.append(csv_path)

    return full_csv_path_list
    

In [8]:
for img in img_list:
    csv_folder = os.path.join(predictions,img)
    csv_list = os.listdir(csv_folder)
    wsi_coords = pd.DataFrame(columns=['x', 'y', 'class'])

    csv_path_list = removeEmptyCsv(csv_folder)
    for csv_path in csv_path_list:
        single_df = pd.read_csv(csv_path, header=None, names=['x', 'y', 'class'])
        # start_x, start_y = (coord * tile_size for coord in coords_from_fname(os.path.basename(csv_path)))
        start_x, start_y = coords_from_fname(os.path.basename(csv_path))

        single_df['x'] = single_df['x'] + start_x
        single_df['y'] = single_df['y'] + start_y

        wsi_coords = pd.concat([wsi_coords, single_df], ignore_index=True)

    wsi_coords.to_csv(os.path.join(parent_output,img+'.csv'), index=False)  # index=False to avoid writing row indices
